# MATH2603 Lab — Preferential Attachment & the BA Model (Advanced)

**Theme (aligns with BA model / scale-free networks lecture):** growth + preferential attachment; degree distributions; power laws; extensions (nonlinear attachment, copying).

**What you will produce today:** a set of plots + short answers in Markdown cells (or your lab log).

> **Tip:** Run cells **top to bottom**. If you get errors like `NameError`, it usually means you skipped a cell.

---

## Learning outcomes
By the end of this lab, you should be able to:
1. Generate ER and BA networks and compare their degree distributions.
2. Explain (in words) why preferential attachment produces hubs.
3. Implement a simple BA-style generator yourself.
4. Explore **nonlinear preferential attachment** (\(k^\alpha\)) and interpret the effect of \(\alpha\).
5. Explore a **copying** mechanism as an alternative route to heavy-tailed degree distributions.
6. (Optional) Estimate a power-law exponent roughly and reflect on limitations.


## 0. Setup check (5–10 min)

Run the next cell. If you get `ModuleNotFoundError`, install the missing package (ask the instructor / use your environment manager).


In [ ]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import random

print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("networkx:", nx.__version__)


## Helper functions (provided)

These functions keep the lab tidy:
- `plot_degree_hist`: histogram of degrees
- `plot_degree_loglog_counts`: log–log counts (not perfect, but good for intuition)
- `ccdf_loglog`: CCDF on log–log axes (often smoother than raw counts)


In [ ]:
def degrees_list(G):
    return np.array([d for _, d in G.degree()], dtype=int)

def plot_degree_hist(deg, bins=30, title="Degree histogram"):
    plt.figure(figsize=(6,4))
    plt.hist(deg, bins=bins)
    plt.xlabel("degree k")
    plt.ylabel("count")
    plt.title(title)
    plt.show()

def plot_degree_loglog_counts(deg, title="Degree counts (log-log)"):
    # count occurrences of each degree value
    vals, counts = np.unique(deg, return_counts=True)
    plt.figure(figsize=(6,4))
    plt.loglog(vals, counts, "o")
    plt.xlabel("degree k")
    plt.ylabel("count")
    plt.title(title)
    plt.show()

def ccdf_loglog(deg, title="CCDF (log-log)"):
    # CCDF: P(K >= k)
    vals, counts = np.unique(deg, return_counts=True)
    N = counts.sum()
    # sort ascending by degree
    order = np.argsort(vals)
    vals = vals[order]
    counts = counts[order]
    # cumulative from the right
    ccdf = np.cumsum(counts[::-1])[::-1] / N
    plt.figure(figsize=(6,4))
    plt.loglog(vals, ccdf, "o")
    plt.xlabel("degree k")
    plt.ylabel("P(K ≥ k)")
    plt.title(title)
    plt.show()

def draw_graph_small(G, title="Graph", seed=0):
    # For small graphs only, otherwise it is too slow/cluttered
    plt.figure(figsize=(6,5))
    pos = nx.spring_layout(G, seed=seed)
    nx.draw(G, pos, node_size=300, with_labels=True)
    plt.title(title)
    plt.show()


## Part A — Warm-up: ER vs BA (basic → intuition)

### Task A1 (ER network)
Generate an Erdős–Rényi (ER) graph \(G(n,p)\) and inspect the degree distribution.

- ER typically has a degree distribution concentrated around its mean (approximately Poisson-like for large \(n\) with small \(p\)).
- Hubs are less extreme than in scale-free networks.

**Do:** Run the cell and look at the plots.


In [ ]:
# Parameters (feel free to tweak)
n = 500
p = 0.02

G_er = nx.erdos_renyi_graph(n, p, seed=0)
deg_er = degrees_list(G_er)

print("ER: n =", G_er.number_of_nodes(), "m =", G_er.number_of_edges())
print("ER: mean degree =", deg_er.mean(), "max degree =", deg_er.max())

plot_degree_hist(deg_er, bins=30, title="ER degree histogram")
plot_degree_loglog_counts(deg_er, title="ER degree counts (log-log)")
ccdf_loglog(deg_er, title="ER CCDF (log-log)")


### Short answer (A1) — write 3–5 sentences

1. Describe the *shape* of the ER degree distribution (roughly symmetric? heavy-tailed?).
2. Did you observe very large hubs? Compare `max degree` to the mean.
3. Why might ER be a poor model for networks like the web or social networks?


### Task A2 (BA network)
Generate a Barabási–Albert (BA) graph with preferential attachment.

- BA tends to produce a **heavy-tailed** degree distribution and visible hubs.
- Parameter \(m\) = number of edges each new node adds.


In [ ]:
n = 500
m = 3

G_ba = nx.barabasi_albert_graph(n, m, seed=0)
deg_ba = degrees_list(G_ba)

print("BA: n =", G_ba.number_of_nodes(), "m =", G_ba.number_of_edges())
print("BA: mean degree =", deg_ba.mean(), "max degree =", deg_ba.max())

plot_degree_hist(deg_ba, bins=30, title="BA degree histogram")
plot_degree_loglog_counts(deg_ba, title="BA degree counts (log-log)")
ccdf_loglog(deg_ba, title="BA CCDF (log-log)")


### Short answer (A2) — write 4–6 sentences

1. Compare BA vs ER: which has bigger hubs, and how do you see this in the plots?
2. On the log–log plots, does BA says “closer to a straight line” than ER? What does that suggest?
3. Why does adding nodes over time (growth) matter for producing hubs?


## Part B — Preferential attachment mechanism (concept → check)

In BA, a new node attaches to existing nodes with probability roughly proportional to their degree:

\[
\Pi(i) = \frac{k_i}{\sum_j k_j}
\]

### Task B1
Compute \(\Pi(i)\) from the current BA graph degrees and inspect how it depends on degree.


In [ ]:
deg = degrees_list(G_ba)
total_degree = deg.sum()
pi = deg / total_degree  # attachment probability up to normalization

# Scatter: (degree, probability)
plt.figure(figsize=(6,4))
plt.scatter(deg, pi, s=10, alpha=0.6)
plt.xlabel("degree k")
plt.ylabel("Pi (proportional to k)")
plt.title("Preferential attachment: Pi vs k (from BA degrees)")
plt.show()

# Also show the average Pi for each degree k (smoother)
vals = np.unique(deg)
avg_pi = np.array([pi[deg == k].mean() for k in vals])

plt.figure(figsize=(6,4))
plt.plot(vals, avg_pi, "o-")
plt.xlabel("degree k")
plt.ylabel("average Pi among nodes with degree k")
plt.title("Average attachment probability by degree")
plt.show()


### Short answer (B1) — write 4–7 sentences

Explain why “rich get richer” emerges from \(\Pi(i) \propto k_i\).  
In your answer, mention:
- what degree represents in a network,
- why high-degree nodes get selected more often,
- how this creates hubs over time.


## Part C — Implement a simple BA generator yourself (coding + understanding)

NetworkX gives you `barabasi_albert_graph`, but implementing a simplified version helps you understand the algorithm.

### Task C1
Implement a BA-style growth process:
1. Start with a small connected seed (often a complete graph of size `m`).
2. For each new node:
   - compute degrees of existing nodes
   - select `m` distinct targets with probability proportional to degree
   - connect the new node to those targets


In [ ]:
def simple_ba(n, m, seed=0):
    rng = np.random.default_rng(seed)
    # Start with a complete graph on m nodes (connected and non-trivial degrees)
    G = nx.complete_graph(m)

    for new_node in range(m, n):
        G.add_node(new_node)

        nodes = np.array(list(G.nodes()))
        deg = np.array([G.degree(v) for v in nodes], dtype=float)

        probs = deg / deg.sum()

        # choose m distinct targets based on probs
        targets = rng.choice(nodes, size=m, replace=False, p=probs)

        for t in targets:
            G.add_edge(new_node, int(t))

    return G

G_ba_manual = simple_ba(n=500, m=3, seed=1)
deg_manual = degrees_list(G_ba_manual)

print("Manual BA: mean degree =", deg_manual.mean(), "max degree =", deg_manual.max())
plot_degree_loglog_counts(deg_manual, title="Manual BA degree counts (log-log)")
ccdf_loglog(deg_manual, title="Manual BA CCDF (log-log)")


### Task C2 (small graph visualisation)
Create a small BA graph and draw it so you can *see* hubs.

> Keep `n_small` small (like 20–40), otherwise the plot gets messy.


In [ ]:
n_small = 25
m_small = 2

G_small = simple_ba(n_small, m_small, seed=2)
draw_graph_small(G_small, title=f"Manual BA (n={n_small}, m={m_small})", seed=2)

deg_s = degrees_list(G_small)
print("Degrees:", list(deg_s))
print("Max degree node(s):", [v for v, d in G_small.degree() if d == deg_s.max()])


### Short answer (C) — write 5–8 sentences

1. Why do we start from a complete graph (or another connected seed) instead of a single node?
2. In your implementation, what happens if you set `replace=True` when choosing targets?
3. What part of the algorithm is responsible for hubs appearing?


## Part D — Nonlinear preferential attachment \(\Pi(k) \propto k^\alpha\) (harder)

Your lecture mentions that the attachment rule can be generalized:

\[
\Pi(i) \propto k_i^{\alpha}
\]

- If **\(\alpha < 1\)**: attachment is *sublinear* → hubs are weaker.
- If **\(\alpha = 1\)**: standard BA.
- If **\(\alpha > 1\)**: *superlinear* → can produce a “winner-takes-all” super-hub.

### Task D1
Implement a nonlinear BA generator and compare outcomes for different \(\alpha\).


In [ ]:
def nonlinear_ba(n, m, alpha=1.0, seed=0):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m)

    for new_node in range(m, n):
        G.add_node(new_node)

        nodes = np.array(list(G.nodes()))
        deg = np.array([G.degree(v) for v in nodes], dtype=float)

        weights = deg ** alpha
        probs = weights / weights.sum()

        targets = rng.choice(nodes, size=m, replace=False, p=probs)
        for t in targets:
            G.add_edge(new_node, int(t))

    return G

n = 800
m = 3
alphas = [0.5, 1.0, 1.5, 2.0]

max_degs = []
plt.figure(figsize=(6,4))
for a in alphas:
    G = nonlinear_ba(n, m, alpha=a, seed=1)
    deg = degrees_list(G)
    max_degs.append(deg.max())
    # plot CCDF for each alpha
    vals, counts = np.unique(deg, return_counts=True)
    N = counts.sum()
    order = np.argsort(vals)
    vals = vals[order]
    counts = counts[order]
    ccdf = np.cumsum(counts[::-1])[::-1] / N
    plt.loglog(vals, ccdf, "o", label=f"alpha={a}")

plt.xlabel("degree k")
plt.ylabel("P(K ≥ k)")
plt.title("Nonlinear preferential attachment: CCDF")
plt.legend()
plt.show()

for a, md in zip(alphas, max_degs):
    print(f"alpha={a}: max degree ≈ {md}")


### Task D2 (challenge)
For one value of \(\alpha\), compute a simple inequality measure of degrees: the **Gini coefficient**.

This is optional but helps quantify “hub dominance”.

> The Gini coefficient is 0 if all degrees are equal; closer to 1 means more unequal.


In [ ]:
def gini(x):
    x = np.array(x, dtype=float)
    x = x[x >= 0]
    if x.size == 0:
        return np.nan
    if np.allclose(x.sum(), 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    cumx = np.cumsum(x)
    # Gini formula using cumulative sums
    g = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n
    return g

# Pick one alpha to study
alpha = 2.0
G = nonlinear_ba(n=800, m=3, alpha=alpha, seed=2)
deg = degrees_list(G)

print("alpha =", alpha)
print("mean degree =", deg.mean())
print("max degree =", deg.max())
print("Gini(degree) =", round(gini(deg), 4))


### Short answer (D) — write 6–10 sentences

1. Describe what changes when \(\alpha\) increases from 0.5 → 2.0.
2. Explain why \(\alpha > 1\) can produce a “winner-takes-all” network.
3. Which \(\alpha\) do you think is most realistic for social networks, and why?


## Part E — Copying model (alternative mechanism) (hard)

Your lecture also mentions link selection / copying ideas:
- A new node “copies” some links from an existing node (e.g., web pages copying reference lists).
- This can also produce heavy-tailed distributions.

### Task E1
Implement a very simple copying model:
1. Start with a small seed graph.
2. For each new node:
   - pick an existing “prototype” node uniformly at random
   - with probability `p_copy`, copy each of its neighbors as an edge
   - if we still have fewer than `m` edges, add random edges to reach `m`


In [ ]:
def copying_model(n, m, p_copy=0.5, seed=0):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m)

    for new_node in range(m, n):
        G.add_node(new_node)

        nodes = list(G.nodes())
        proto = int(rng.choice(nodes))  # prototype node
        neighbors = list(G.neighbors(proto))

        selected = []
        for nb in neighbors:
            if rng.random() < p_copy:
                selected.append(int(nb))

        # If too few copied links, add random nodes (avoid self-loops and duplicates)
        while len(selected) < m:
            candidate = int(rng.choice(nodes))
            if candidate != new_node and candidate not in selected:
                selected.append(candidate)

        # Keep exactly m targets
        selected = selected[:m]
        for t in selected:
            if t != new_node:
                G.add_edge(new_node, t)

    return G

n = 800
m = 3
p_list = [0.2, 0.5, 0.8]

plt.figure(figsize=(6,4))
for p_copy in p_list:
    Gc = copying_model(n, m, p_copy=p_copy, seed=1)
    deg = degrees_list(Gc)
    # CCDF
    vals, counts = np.unique(deg, return_counts=True)
    N = counts.sum()
    order = np.argsort(vals)
    vals = vals[order]
    counts = counts[order]
    ccdf = np.cumsum(counts[::-1])[::-1] / N
    plt.loglog(vals, ccdf, "o", label=f"p_copy={p_copy}")

plt.xlabel("degree k")
plt.ylabel("P(K ≥ k)")
plt.title("Copying model: CCDF for different p_copy")
plt.legend()
plt.show()


### Short answer (E) — write 6–10 sentences

1. How does increasing `p_copy` change the tail of the distribution?
2. Compare copying vs preferential attachment: what is similar, and what is different?
3. Give one real-world example where copying might be a more realistic mechanism than “rich-get-richer”.


## Part F — (Optional) Rough exponent estimation (very optional, reflective)

**Warning:** Estimating power-law exponents properly is non-trivial.  
Here we do a *rough* check using a log–log linear fit on the CCDF tail.

### Task F1
Pick a BA-like network and fit a line to the tail of log(CCDF) vs log(k).

> Do not over-interpret the result. This is for intuition only.


In [ ]:
def rough_ccdf_tail_slope(deg, k_min=5):
    vals, counts = np.unique(deg, return_counts=True)
    N = counts.sum()
    order = np.argsort(vals)
    vals = vals[order]
    counts = counts[order]
    ccdf = np.cumsum(counts[::-1])[::-1] / N

    # Tail selection
    mask = vals >= k_min
    x = np.log(vals[mask])
    y = np.log(ccdf[mask])

    if x.size < 2:
        return np.nan

    # Linear fit: y = a x + b
    a, b = np.polyfit(x, y, 1)
    return a, b, vals, ccdf, mask

G = nx.barabasi_albert_graph(2000, 3, seed=0)
deg = degrees_list(G)

a, b, vals, ccdf, mask = rough_ccdf_tail_slope(deg, k_min=6)

plt.figure(figsize=(6,4))
plt.loglog(vals, ccdf, "o", alpha=0.6, label="CCDF")
# plot fitted line on tail
x_tail = np.log(vals[mask])
y_fit = a * x_tail + b
plt.loglog(vals[mask], np.exp(y_fit), "-", label=f"fit slope≈{a:.2f}")
plt.xlabel("degree k")
plt.ylabel("P(K ≥ k)")
plt.title("Rough CCDF tail fit (for intuition)")
plt.legend()
plt.show()

print("Rough slope (CCDF) a ≈", round(a, 3))
print("Note: For a power-law PDF with exponent gamma, CCDF slope is approximately -(gamma-1).")


## Wrap-up reflection (5–10 min)

Write 6–10 sentences reflecting on:

1. Which mechanism(s) in this lab could plausibly generate real-world networks (web, citations, social)?
2. What *limitations* do these models have (e.g., direction, weights, node/edge deletion, communities)?
3. If you were to improve the BA model for a real dataset, what would you add?

---
**End of lab**
